In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Modul 6b: Pipeline

Cerita lengkapnya ada di `README.md`. Di sini: hitung dulu, baru jalankan kode.

Tiga gelas latih: 200, 50, dan 180. Satu gelas uji: 40.

1. Bolehkah 40 ikut menyetel rata-rata timbangan?
2. Berapa jumlah tiga gelas latih?
3. Berapa rata-ratanya?

Hitung dengan tangan, lalu jalankan sel dari atas ke bawah.


In [2]:
# Pustaka adalah paket kode siap pakai yang kita panggil.
# Array, istilah modul persiapan: kumpulan nilai sejenis dengan bentuk yang tetap, yang dihitung sekaligus.
# Satu ide: panggil NumPy, pustaka untuk hitungan pada array.
import numpy as np


In [3]:
# DataFrame, istilah modul persiapan: tabel di memori, setiap kolom punya nama, setiap baris satu catatan.
# Satu ide: panggil pandas, pustaka untuk tabel.
import pandas as pd


In [4]:
# Satu ide: jumlah dan rata-rata tiga gelas latih. Gelas 40 tidak ikut.
latih_ml = np.array([200, 50, 180])
print("jumlah_latih", int(latih_ml.sum()))
print("rata_latih", float(latih_ml.mean()))


jumlah_latih 430
rata_latih 143.33333333333334


## Kasus nyata

Encoding mengubah kategori, yaitu teks berlabel terbatas, menjadi angka supaya model bisa menghitung. Penskalaan mengubah angka sebuah kolom supaya rentangnya sebanding dengan kolom lain, memakai rata-rata dan sebaran dari data yang boleh dilihat. Keduanya istilah bagian rekayasa fitur, bukan istilah baru di sini.

Model adalah cara menebak yang belajar dari data latih. Label adalah jawaban yang ingin ditebak pada tiap baris.

Tabel dibuat di memori. Tidak ada unduhan. Ini bukan data asli perusahaan. Aturan main yang kita tanam: `kembali` berisi ya bila `gelas_ml` di bawah 100. Kolom `catatan_pinggir` hanya coretan, bukan fitur.


In [5]:
# Satu ide: buat empat puluh baris. default_rng memakai SEED supaya tabel ini berulang.
rng = np.random.default_rng(SEED)
n = 40
gelas_ml = rng.integers(30, 220, size=n)
suhu = np.where(rng.random(n) < 0.5, "dingin", "panas")
kembali = np.where(gelas_ml < 100, "ya", "tidak")
kasus = pd.DataFrame(
    {
        "gelas_ml": gelas_ml,
        "suhu": suhu,
        "catatan_pinggir": "gelas",
        "kembali": kembali,
    }
)
print("n_baris", len(kasus))


n_baris 40


In [6]:
# train_test_split membagi baris menjadi data latih dan data uji.
# Kocokan dikunci dengan random_state=SEED.
# Satu ide: panggil pembagi itu.
from sklearn.model_selection import train_test_split


In [7]:
# Satu ide: bagi tabel. Jawaban tidak masuk ke X.
X = kasus[["gelas_ml", "suhu", "catatan_pinggir"]]
y = kasus["kembali"]
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y, test_size=0.25, random_state=SEED
)
print("n_latih", len(X_latih))
print("n_uji", len(X_uji))


n_latih 30
n_uji 10


In [8]:
# ColumnTransformer menerapkan cara ubah yang berbeda pada kelompok kolom yang berbeda.
# remainder="drop" membuang kolom yang tidak disebut. Itu nilai bawaannya, ditulis di sini supaya terlihat.
# Satu ide: panggil ColumnTransformer dan dua pengubah kolom.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [9]:
# Satu ide: angka diskalakan, kategori di-encode, coretan tidak disebut.
penggabung = ColumnTransformer(
    transformers=[
        ("angka", StandardScaler(), ["gelas_ml"]),
        (
            "kategori",
            OneHotEncoder(sparse_output=False),
            ["suhu"],
        ),
    ],
    remainder="drop",
)


In [10]:
# Regresi logistik adalah model yang menebak salah satu dari dua label.
# Namanya dari modul klasifikasi. Di sini ia hanya berdiri di ujung rantai.
# Satu ide: panggil Pipeline dan model di ujungnya.
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


In [11]:
# Pipeline adalah satu rantai langkah berurutan.
# Praolah adalah langkah yang mengubah data sebelum model.
# Satu ide: susun rantainya.
pipa = Pipeline(
    steps=[
        ("praolah", penggabung),
        ("model", LogisticRegression(random_state=SEED)),
    ]
)
print("langkah", list(pipa.named_steps))


langkah ['praolah', 'model']


In [12]:
# fit pada Pipeline belajar dari data latih saja, termasuk penskalaan di dalam rantai.
# Satu ide: belajar, lalu tebak data uji.
pipa.fit(X_latih, y_latih)
tebakan_1 = pipa.predict(X_uji)
print("tebakan", tebakan_1.tolist())
assert float(
    pipa.named_steps["praolah"].named_transformers_["angka"].mean_[0]
) == float(X_latih["gelas_ml"].mean()), (
    "Rata-rata di dalam rantai harus sama dengan rata-rata data latih."
)


tebakan ['tidak', 'tidak', 'tidak', 'tidak', 'tidak', 'tidak', 'tidak', 'tidak', 'tidak', 'ya']


In [13]:
# Satu ide: ulangi rantai yang sama. Tebakannya harus sama.
pipa.fit(X_latih, y_latih)
tebakan_2 = pipa.predict(X_uji)
print("sama", tebakan_1.tolist() == tebakan_2.tolist())
assert float(
    pipa.named_steps["praolah"].named_transformers_["angka"].mean_[0]
) == float(X_latih["gelas_ml"].mean()), (
    "Rata-rata di dalam rantai harus sama dengan rata-rata data latih."
)


sama True


In [14]:
# Nama kolom hasil diawali nama langkah, supaya asal kolomnya ketahuan.
# Satu ide: lihat nama hasil, dan cek apakah coretan ikut.
nama_fitur = pipa.named_steps["praolah"].get_feature_names_out().tolist()
print("nama_fitur", nama_fitur)
print("ada_catatan", any("catatan_pinggir" in nama for nama in nama_fitur))


nama_fitur ['angka__gelas_ml', 'kategori__suhu_dingin', 'kategori__suhu_panas']
ada_catatan False


In [15]:
# named_steps mengambil langkah bernama. named_transformers_ mengambil pengubah kolom di dalamnya.
# Satu ide: rata-rata di dalam rantai hanya dari data latih, bukan dari seluruh kolom.
rata_dalam_rantai = float(
    pipa.named_steps["praolah"].named_transformers_["angka"].mean_[0]
)
rata_semua = float(kasus["gelas_ml"].mean())
beda = rata_dalam_rantai != rata_semua
print("rata_dalam_rantai", rata_dalam_rantai)
print("rata_semua", rata_semua)
print("beda", beda)
assert beda is True, "Rata-rata latih harus berbeda dari rata-rata seluruh kolom."
assert rata_dalam_rantai == float(X_latih["gelas_ml"].mean()), (
    "Rata-rata di dalam rantai harus sama dengan rata-rata data latih."
)


rata_dalam_rantai 128.16666666666666
rata_semua 130.55
beda True


In [16]:
# Satu ide: hitung berapa tebakan yang cocok dengan label uji.
# Aturan mainnya kita yang menanam, jadi angka ini bukan bukti untuk data lain.
cocok = int((tebakan_1 == y_uji.to_numpy()).sum())
print("cocok", cocok)
print("dari", len(y_uji))


cocok 10
dari 10
